<a href="https://colab.research.google.com/github/Datadog-995/Dirty-Financial-Transactions--Data-Integrity/blob/main/Messy_to_CLEAN_FINANCIAL_TRANSACTIONS_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import pandas as pd
import numpy as np

# Load the financial transactions data
file_path = '/content/drive/MyDrive/NEW-dirty_financial_transactions.xlsx'
df = pd.read_excel(file_path)

# We will insert the new column at the position immediately after 'Transaction_Date'

# Get the index of the column
col_index = df.columns.get_loc('Transaction_Date')

# Create the new column at that index + 1
df.insert(col_index + 1, 'audit Trans_date', None)

# Convert 'Transaction_Date' to datetime, coercing errors to NaT
df['Transaction_Date_cleaned'] = pd.to_datetime(df['Transaction_Date'], errors='coerce')

# Fill 'audit Trans_date' based on validity and missingness
df['audit Trans_date'] = np.where(
    df['Transaction_Date_cleaned'].isna() & df['Transaction_Date'].notna(),
    'Invalid',
    np.where(df['Transaction_Date'].isna(), 'Missing', None)
)

# Drop the temporary cleaned date column
df = df.drop(columns=['Transaction_Date_cleaned'])

# Display the first few rows to verify
print(df.head())

  Transaction_ID     Transaction_Date audit Trans_date Customer_ID  \
0          T0001  2024-08-02 00:00:00             None       C2205   
1          T0002  2020-02-10 00:00:00             None       C3156   
2          T0003           2025-02-30          Invalid       C2919   
3          T0004  2020-08-17 00:00:00             None       C3009   
4          T0005           2025-02-30          Invalid       C3488   

     Product_Name  Quantity       Price Payment_Method Transaction_Status  
0      Headphones      -5.0  420.210000        pay pal                NaN  
1          Coffee     469.0 -445.342025     creditcard            Pending  
2          Tablet      -4.0  810.993012    credit card          completed  
3             Tab      -7.0  868.608341         PayPal            Pending  
4  Coffee Machine     -10.0 -763.122449         PayPal          completed  


In [9]:
# Define the mapping based on the first letter of the product name
product_name_mapping = {
    'h': 'Headphones',
    'c': 'Coffee Machine',
    't': 'Tablet',
    's': 'Smartphone',
    'l': 'Laptop' # Added mapping for 'L' to 'Laptop'
}

def standardize_product_name(name):
    if pd.isna(name): # Handle NaN values
        return name
    first_letter = str(name).lower()[0]
    return product_name_mapping.get(first_letter, name) # Keep original if no match

# Apply the standardization to the 'Product_Name' column
df['Product_Name'] = df['Product_Name'].apply(standardize_product_name)

# Display the updated first 5 rows to verify the changes
print(df.head())

  Transaction_ID     Transaction_Date audit Trans_date Customer_ID  \
0          T0001  2024-08-02 00:00:00             None       C2205   
1          T0002  2020-02-10 00:00:00             None       C3156   
2          T0003           2025-02-30          Invalid       C2919   
3          T0004  2020-08-17 00:00:00             None       C3009   
4          T0005           2025-02-30          Invalid       C3488   

     Product_Name  Quantity       Price Payment_Method Transaction_Status  
0      Headphones      -5.0  420.210000        pay pal                NaN  
1  Coffee Machine     469.0 -445.342025     creditcard            Pending  
2          Tablet      -4.0  810.993012    credit card          completed  
3          Tablet      -7.0  868.608341         PayPal            Pending  
4  Coffee Machine     -10.0 -763.122449         PayPal          completed  


In [10]:
unique_product_names = df['Product_Name'].unique()
print(unique_product_names)

['Headphones' 'Coffee Machine' 'Tablet' 'Smartphone' 'Laptop']


In [11]:
# Get the index of the 'Quantity' column
col_index_quantity = df.columns.get_loc('Quantity')

# Insert the new 'audit Quantity' column after 'Quantity'
df.insert(col_index_quantity + 1, 'audit Quantity', None)

# Flag negative numbers in 'audit Quantity'
df.loc[df['Quantity'] < 0, 'audit Quantity'] = 'negative'

# Display the first few rows to verify the changes
print(df.head())

  Transaction_ID     Transaction_Date audit Trans_date Customer_ID  \
0          T0001  2024-08-02 00:00:00             None       C2205   
1          T0002  2020-02-10 00:00:00             None       C3156   
2          T0003           2025-02-30          Invalid       C2919   
3          T0004  2020-08-17 00:00:00             None       C3009   
4          T0005           2025-02-30          Invalid       C3488   

     Product_Name  Quantity audit Quantity       Price Payment_Method  \
0      Headphones      -5.0       negative  420.210000        pay pal   
1  Coffee Machine     469.0           None -445.342025     creditcard   
2          Tablet      -4.0       negative  810.993012    credit card   
3          Tablet      -7.0       negative  868.608341         PayPal   
4  Coffee Machine     -10.0       negative -763.122449         PayPal   

  Transaction_Status  
0                NaN  
1            Pending  
2          completed  
3            Pending  
4          completed  


In [12]:
import numpy as np

# Get the index of the 'Price' column
col_index_price = df.columns.get_loc('Price')

# Insert the new 'audit Price' column after 'Price'
df.insert(col_index_price + 1, 'audit Price', None)

# Flag negative numbers in 'audit Price'
df.loc[df['Price'] < 0, 'audit Price'] = 'negative'

# Display the first few rows to verify the changes
print(df.head())

  Transaction_ID     Transaction_Date audit Trans_date Customer_ID  \
0          T0001  2024-08-02 00:00:00             None       C2205   
1          T0002  2020-02-10 00:00:00             None       C3156   
2          T0003           2025-02-30          Invalid       C2919   
3          T0004  2020-08-17 00:00:00             None       C3009   
4          T0005           2025-02-30          Invalid       C3488   

     Product_Name  Quantity audit Quantity       Price audit Price  \
0      Headphones      -5.0       negative  420.210000        None   
1  Coffee Machine     469.0           None -445.342025    negative   
2          Tablet      -4.0       negative  810.993012        None   
3          Tablet      -7.0       negative  868.608341        None   
4  Coffee Machine     -10.0       negative -763.122449    negative   

  Payment_Method Transaction_Status  
0        pay pal                NaN  
1     creditcard            Pending  
2    credit card          completed  
3     

In [13]:
# Trim whitespace from all string columns
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.strip()

# Display the first few rows to verify the changes
print(df.head())

  Transaction_ID     Transaction_Date audit Trans_date Customer_ID  \
0          T0001  2024-08-02 00:00:00             None       C2205   
1          T0002  2020-02-10 00:00:00             None       C3156   
2          T0003           2025-02-30          Invalid       C2919   
3          T0004  2020-08-17 00:00:00             None       C3009   
4          T0005           2025-02-30          Invalid       C3488   

     Product_Name  Quantity audit Quantity       Price audit Price  \
0      Headphones      -5.0       negative  420.210000        None   
1  Coffee Machine     469.0           None -445.342025    negative   
2          Tablet      -4.0       negative  810.993012        None   
3          Tablet      -7.0       negative  868.608341        None   
4  Coffee Machine     -10.0       negative -763.122449    negative   

  Payment_Method Transaction_Status  
0        pay pal                nan  
1     creditcard            Pending  
2    credit card          completed  
3     

In [14]:
# Format the 'Price' column to two decimal places
df['Price'] = df['Price'].round(2)

# Display the first few rows to verify the changes
print(df.head())

  Transaction_ID     Transaction_Date audit Trans_date Customer_ID  \
0          T0001  2024-08-02 00:00:00             None       C2205   
1          T0002  2020-02-10 00:00:00             None       C3156   
2          T0003           2025-02-30          Invalid       C2919   
3          T0004  2020-08-17 00:00:00             None       C3009   
4          T0005           2025-02-30          Invalid       C3488   

     Product_Name  Quantity audit Quantity   Price audit Price Payment_Method  \
0      Headphones      -5.0       negative  420.21        None        pay pal   
1  Coffee Machine     469.0           None -445.34    negative     creditcard   
2          Tablet      -4.0       negative  810.99        None    credit card   
3          Tablet      -7.0       negative  868.61        None         PayPal   
4  Coffee Machine     -10.0       negative -763.12    negative         PayPal   

  Transaction_Status  
0                nan  
1            Pending  
2          completed  


In [15]:
# Function to standardize payment methods
def standardize_payment_method(method):
    if pd.isna(method):
        return None
    method_lower = str(method).lower().strip()
    if 'credit' in method_lower:
        return 'Credit Card'
    elif 'pay pal' in method_lower or 'paypal' in method_lower:
        return 'PayPal'
    elif 'cash' in method_lower:
        return 'Cash'
    else:
        return method  # Return original if no match

# Apply the standardization to the 'Payment_Method' column
df['Payment_Method'] = df['Payment_Method'].apply(standardize_payment_method)

# Display the updated first few rows and unique payment methods to verify
print(df.head())
print("\nUnique Payment Methods after standardization:")
print(df['Payment_Method'].unique())

  Transaction_ID     Transaction_Date audit Trans_date Customer_ID  \
0          T0001  2024-08-02 00:00:00             None       C2205   
1          T0002  2020-02-10 00:00:00             None       C3156   
2          T0003           2025-02-30          Invalid       C2919   
3          T0004  2020-08-17 00:00:00             None       C3009   
4          T0005           2025-02-30          Invalid       C3488   

     Product_Name  Quantity audit Quantity   Price audit Price Payment_Method  \
0      Headphones      -5.0       negative  420.21        None         PayPal   
1  Coffee Machine     469.0           None -445.34    negative    Credit Card   
2          Tablet      -4.0       negative  810.99        None    Credit Card   
3          Tablet      -7.0       negative  868.61        None         PayPal   
4  Coffee Machine     -10.0       negative -763.12    negative         PayPal   

  Transaction_Status  
0                nan  
1            Pending  
2          completed  


In [17]:
# Function to standardize transaction status
def standardize_transaction_status(status):
    status_str = str(status).lower().strip()
    if status_str == 'nan' or pd.isna(status): # Handle both NaN and string 'nan'
        return None
    elif 'complete' in status_str: # Catches 'completed' and 'complete'
        return 'Completed'
    elif 'pending' in status_str:
        return 'Pending'
    elif 'fail' in status_str: # Catches 'failed'
        return 'Failed'
    else:
        return status  # Return original if no specific match

# Apply the standardization to the 'Transaction_Status' column
df['Transaction_Status'] = df['Transaction_Status'].apply(standardize_transaction_status)

# Display the updated first few rows and unique transaction statuses to verify
print(df.head())
print("\nUnique Transaction Statuses after standardization:")
print(df['Transaction_Status'].unique())

  Transaction_ID     Transaction_Date audit Trans_date Customer_ID  \
0          T0001  2024-08-02 00:00:00             None       C2205   
1          T0002  2020-02-10 00:00:00             None       C3156   
2          T0003           2025-02-30          Invalid       C2919   
3          T0004  2020-08-17 00:00:00             None       C3009   
4          T0005           2025-02-30          Invalid       C3488   

     Product_Name  Quantity audit Quantity   Price audit Price Payment_Method  \
0      Headphones      -5.0       negative  420.21        None         PayPal   
1  Coffee Machine     469.0           None -445.34    negative    Credit Card   
2          Tablet      -4.0       negative  810.99        None    Credit Card   
3          Tablet      -7.0       negative  868.61        None         PayPal   
4  Coffee Machine     -10.0       negative -763.12    negative         PayPal   

  Transaction_Status  
0               None  
1            Pending  
2          Completed  


In [18]:
# Convert 'Transaction_Date' to datetime objects. Invalid dates are already flagged by 'audit Trans_date', so we can coerce errors.
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], errors='coerce')

# Convert 'Quantity' to numeric, coercing errors
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

# Check data types after conversion
print("Data types after conversion:")
print(df.info())

Data types after conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Transaction_ID      100000 non-null  object        
 1   Transaction_Date    31739 non-null   datetime64[ns]
 2   audit Trans_date    100000 non-null  object        
 3   Customer_ID         100000 non-null  object        
 4   Product_Name        100000 non-null  object        
 5   Quantity            94981 non-null   float64       
 6   audit Quantity      100000 non-null  object        
 7   Price               66503 non-null   float64       
 8   audit Price         100000 non-null  object        
 9   Payment_Method      100000 non-null  object        
 10  Transaction_Status  83321 non-null   object        
dtypes: datetime64[ns](1), float64(2), object(8)
memory usage: 8.4+ MB
None


In [19]:
# Check for remaining missing values across the DataFrame
print("\nMissing values per column after conversions:")
print(df.isnull().sum())


Missing values per column after conversions:
Transaction_ID            0
Transaction_Date      68261
audit Trans_date          0
Customer_ID               0
Product_Name              0
Quantity               5019
audit Quantity            0
Price                 33497
audit Price               0
Payment_Method            0
Transaction_Status    16679
dtype: int64


In [20]:
# Display the first few rows to verify the changes and types
print("\nDataFrame head after type conversions:")
print(df.head())


DataFrame head after type conversions:
  Transaction_ID Transaction_Date audit Trans_date Customer_ID  \
0          T0001       2024-08-02             None       C2205   
1          T0002       2020-02-10             None       C3156   
2          T0003              NaT          Invalid       C2919   
3          T0004       2020-08-17             None       C3009   
4          T0005              NaT          Invalid       C3488   

     Product_Name  Quantity audit Quantity   Price audit Price Payment_Method  \
0      Headphones      -5.0       negative  420.21        None         PayPal   
1  Coffee Machine     469.0           None -445.34    negative    Credit Card   
2          Tablet      -4.0       negative  810.99        None    Credit Card   
3          Tablet      -7.0       negative  868.61        None         PayPal   
4  Coffee Machine     -10.0       negative -763.12    negative         PayPal   

  Transaction_Status  
0               None  
1            Pending  
2      

In [21]:
# Impute missing 'Quantity' with the median
median_quantity = df['Quantity'].median()
df['Quantity'] = df['Quantity'].fillna(median_quantity)

# Impute missing 'Price' with the median
median_price = df['Price'].median()
df['Price'] = df['Price'].fillna(median_price)

# Impute missing 'Transaction_Status' with 'Unknown'
df['Transaction_Status'] = df['Transaction_Status'].fillna('Unknown')

# Display missing values after imputation to verify
print("\nMissing values per column after imputation:")
print(df.isnull().sum())

# Display the first few rows to verify the changes
print("\nDataFrame head after imputation:")
print(df.head())


Missing values per column after imputation:
Transaction_ID            0
Transaction_Date      68261
audit Trans_date          0
Customer_ID               0
Product_Name              0
Quantity                  0
audit Quantity            0
Price                     0
audit Price               0
Payment_Method            0
Transaction_Status        0
dtype: int64

DataFrame head after imputation:
  Transaction_ID Transaction_Date audit Trans_date Customer_ID  \
0          T0001       2024-08-02             None       C2205   
1          T0002       2020-02-10             None       C3156   
2          T0003              NaT          Invalid       C2919   
3          T0004       2020-08-17             None       C3009   
4          T0005              NaT          Invalid       C3488   

     Product_Name  Quantity audit Quantity   Price audit Price Payment_Method  \
0      Headphones      -5.0       negative  420.21        None         PayPal   
1  Coffee Machine     469.0           Non

In [22]:
# Define the path for the new cleaned CSV file in Google Drive
output_file_path = '/content/drive/MyDrive/cleaned_financial_transactions.csv'

# Save the DataFrame to a CSV file, without the index
df.to_csv(output_file_path, index=False)

print(f"Cleaned DataFrame saved successfully to: {output_file_path}")

Cleaned DataFrame saved successfully to: /content/drive/MyDrive/cleaned_financial_transactions.csv


In [23]:
# Define the content of the summary as a multi-line string
summary_content = """
## Data Cleaning and Preparation Steps

This document outlines the data cleaning and preparation process applied to the `NEW-dirty_financial_transactions.xlsx` dataset. The goal was to enhance data quality, consistency, and usability for further analysis.

### 1. Initial Data Loading and Mounting Google Drive

**Purpose:** To load the raw dataset into a pandas DataFrame and ensure access to files stored in Google Drive.

**Code Used:**
*   Mounting Google Drive: `from google.colab import drive; drive.mount('/content/drive')`
*   Loading Excel file: `df = pd.read_excel(file_path)`

### 2. Auditing and Cleaning `Transaction_Date` Column

**Purpose:** Identify and flag invalid or missing dates within the `Transaction_Date` column to ensure data integrity and enable proper datetime operations.

**Details:**
*   A new column, `audit Trans_date`, was inserted immediately after `Transaction_Date`.
*   Invalid dates (e.g., '2025-02-30') were identified and flagged as 'Invalid'.
*   Missing dates were flagged as 'Missing'.
*   The original `Transaction_Date` was converted to `datetime` objects, coercing invalid entries to `NaT` (Not a Time).

**Code Used:**
*   `df.insert()`, `pd.to_datetime()`, `np.where()`, `df.drop()`

### 3. Standardizing `Product_Name` Column

**Purpose:** Ensure consistency in product names by mapping variations to a standardized format.

**Details:**
*   A mapping dictionary was created (e.g., 'H' to 'Headphones', 'C' to 'Coffee Machine', 'T' to 'Tablet', 'S' to 'Smartphone', 'L' to 'Laptop').
*   A function was applied to the `Product_Name` column to standardize entries based on their first letter.

**Code Used:**
*   Custom Python function with `str.lower()`, `str.strip()`, `dict.get()`, and `df.apply()`.

### 4. Auditing `Quantity` Column

**Purpose:** Identify and flag rows where `Quantity` values are negative, indicating potential data entry errors or specific transaction types needing attention.

**Details:**
*   A new column, `audit Quantity`, was inserted next to `Quantity`.
*   Rows with `Quantity < 0` were flagged as 'negative'.

**Code Used:**
*   `df.insert()`, `df.loc[]` for conditional assignment.

### 5. Auditing `Price` Column

**Purpose:** Similar to `Quantity`, flag negative `Price` values.

**Details:**
*   A new column, `audit Price`, was inserted next to `Price`.
*   Rows with `Price < 0` were flagged as 'negative'.

**Code Used:**
*   `df.insert()`, `df.loc[]` for conditional assignment.

### 6. Trimming Whitespace from All String Columns

**Purpose:** Remove leading and trailing whitespace from all object (string) type columns to prevent inconsistencies in categorical data.

**Details:**
*   Iterated through all columns with `object` dtype.
*   Applied `.str.strip()` to each string entry.

**Code Used:**
*   Loop with `df.select_dtypes(include=['object']).columns`, `df[col].astype(str).str.strip()`.

### 7. Formatting `Price` Column to Two Decimal Places

**Purpose:** Standardize the display and precision of monetary values in the `Price` column.

**Details:**
*   Rounded all values in the `Price` column to two decimal places.

**Code Used:**
*   `df['Price'].round(2)`

### 8. Standardizing `Payment_Method` Column

**Purpose:** Consolidate various spellings and formats of payment methods into consistent categories.

**Details:**
*   A function was created to map variations like 'pay pal', 'creditcard', 'credit card' to 'PayPal', 'Credit Card', or 'Cash'.

**Code Used:**
*   Custom Python function with `str.lower()`, `str.strip()`, conditional logic (`if 'credit' in...`), and `df.apply()`.

### 9. Standardizing `Transaction_Status` Column

**Purpose:** Unify transaction status entries into a clean set of categories and handle missing values.

**Details:**
*   A function was created to map variations like 'completed', 'complete', 'pending', 'failed', and 'nan' to 'Completed', 'Pending', 'Failed', or `None`.
*   Missing values were later imputed to 'Unknown'.

**Code Used:**
*   Custom Python function with `str.lower()`, `str.strip()`, `pd.isna()`, conditional logic, and `df.apply()`.

### 10. Data Type Conversion for `Quantity` and `Transaction_Date`

**Purpose:** Ensure numerical and temporal columns have appropriate data types for accurate calculations and time-series analysis.

**Details:**
*   `Transaction_Date` was explicitly converted to `datetime64[ns]`, with errors coerced to `NaT`.
*   `Quantity` was explicitly converted to `float64`, with errors coerced to `NaN`.

**Code Used:**
*   `pd.to_datetime(errors='coerce')`, `pd.to_numeric(errors='coerce')`.

### 11. Imputing Remaining Missing Values

**Purpose:** Fill in any lingering missing values in key columns to ensure the dataset is complete for analysis.

**Details:**
*   Missing `Quantity` values were filled with the *median* of the `Quantity` column.
*   Missing `Price` values were filled with the *median* of the `Price` column.
*   Missing `Transaction_Status` values (which were `None` after standardization) were filled with 'Unknown'.

**Code Used:**
*   `df['Column'].median()`, `df['Column'].fillna()`

### 12. Saving the Cleaned Dataset

**Purpose:** Export the thoroughly cleaned and prepared DataFrame to a new file for future use and sharing.

**Details:**
*   The cleaned DataFrame was saved as `cleaned_financial_transactions.csv` to Google Drive.

**Code Used:**
*   `df.to_csv(output_file_path, index=False)`
"""

# Define the path for the summary Markdown file in Google Drive
summary_file_path = '/content/drive/MyDrive/Data_Cleaning_Summary.md'

# Write the summary content to the Markdown file
with open(summary_file_path, 'w') as f:
    f.write(summary_content)

print(f"Data cleaning summary saved successfully to: {summary_file_path}")

Data cleaning summary saved successfully to: /content/drive/MyDrive/Data_Cleaning_Summary.md
